[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ImagingDataCommons/CloudSegmentator/blob/main/workflows/MOOSE/Notebooks/endToEndMOOSENotebook.ipynb)

# **This Notebook downloads CT data from Imaging Data Commons, runs MOOSE (moosez) multi-organ segmentation, and produces compressed NIfTI segmentation masks**

Please cite:

Shiyam Sundar LK, et al. Fully automated, semantic segmentation of whole-body 18F-FDG PET/CT images based on data-centric artificial intelligence. J Nucl Med. 2022. https://doi.org/10.2967/jnumed.122.264063

Isensee, F., Jaeger, P.F., Kohl, S.A.A. et al. nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nat Methods 18, 203–211 (2021). https://doi.org/10.1038/s41592-020-01008-z

Li X, Morgan PS, Ashburner J, Smith J, Rorden C. (2016) The first step for neuroimaging data analysis: DICOM to NIfTI conversion. J Neurosci Methods. 264:47-56.

## Setup

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

import nibabel as nib
import nvidia_smi
import yaml
from idc_index.index import IDCClient
from p_tqdm import p_map

## Parameters

The cell below is tagged `parameters` so papermill can inject values at runtime.
When running interactively, edit the YAML string directly.

In [ ]:
# papermill parameters — these defaults are overridden at runtime via -y
# SeriesInstanceUIDs is a list when injected by papermill from YAML
SeriesInstanceUIDs = ["1.3.6.1.4.1.14519.5.2.1.7009.9004.100143549999116733615345241533"]

# Comma-separated list of moosez model names to run
moose_models = "clin_ct_organs,clin_ct_ribs,clin_ct_vertebrae"

# Accelerator: 'cuda' for GPU, 'cpu' for CPU-only
accelerator = "cuda"

## Parse parameters

In [ ]:
# papermill injects SeriesInstanceUIDs as a Python list when using -y "SeriesInstanceUIDs: [...]"
# Handle both list (papermill injection) and string (legacy / interactive) forms.
if isinstance(SeriesInstanceUIDs, str):
    parsed = yaml.safe_load(SeriesInstanceUIDs)
    series_uids = parsed.get("SeriesInstanceUIDs", parsed) if isinstance(parsed, dict) else parsed
elif isinstance(SeriesInstanceUIDs, list):
    series_uids = SeriesInstanceUIDs
else:
    series_uids = list(SeriesInstanceUIDs)

if isinstance(series_uids, str):
    series_uids = [series_uids]

models = [m.strip() for m in moose_models.split(",") if m.strip()]

print(f"Series to process : {len(series_uids)}")
print(f"MOOSE models       : {models}")
print(f"Accelerator        : {accelerator}")

## GPU availability check

In [ ]:
usage_metrics = {"gpu": [], "series": {}}

try:
    nvidia_smi.nvmlInit()
    gpu_count = nvidia_smi.nvmlDeviceGetCount()
    for i in range(gpu_count):
        handle = nvidia_smi.nvmlDeviceGetHandleByIndex(i)
        name = nvidia_smi.nvmlDeviceGetName(handle)
        mem = nvidia_smi.nvmlDeviceGetMemoryInfo(handle)
        usage_metrics["gpu"].append({
            "index": i,
            "name": name if isinstance(name, str) else name.decode(),
            "total_mb": mem.total // (1024 ** 2),
        })
    print(f"GPU(s) found: {usage_metrics['gpu']}")
except Exception as exc:
    print(f"No GPU detected or nvidia-smi unavailable: {exc}")
    if accelerator == "cuda":
        print("WARNING: accelerator=cuda but no GPU found; falling back to cpu")
        accelerator = "cpu"

## Download DICOM from IDC

In [ ]:
DICOM_DIR = Path("/tmp/dicom")
DICOM_DIR.mkdir(parents=True, exist_ok=True)

client = IDCClient()
download_errors = []

for uid in series_uids:
    dest = DICOM_DIR / uid
    dest.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    try:
        client.download_from_selection(
            downloadDir=str(dest),
            seriesInstanceUID=uid,
        )
        usage_metrics["series"].setdefault(uid, {})["download_s"] = round(time.time() - t0, 1)
        print(f"Downloaded {uid} in {usage_metrics['series'][uid]['download_s']}s")
    except Exception as exc:
        msg = f"{uid}: {traceback.format_exc()}"
        download_errors.append(msg)
        print(f"ERROR downloading {uid}: {exc}")

if download_errors:
    Path("download_error_file.txt").write_text("\n".join(download_errors))

## Convert DICOM → NIfTI with dcm2niix

In [ ]:
NIFTI_DIR = Path("/tmp/nifti")
NIFTI_DIR.mkdir(parents=True, exist_ok=True)

dcm2niix_errors = []
converted_uids = []

for uid in series_uids:
    if uid in [e.split(":")[0] for e in download_errors]:
        continue  # skip series that failed to download

    dcm_path = DICOM_DIR / uid
    nii_path = NIFTI_DIR / uid
    nii_path.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    result = subprocess.run(
        ["dcm2niix", "-z", "y", "-f", "%i_%s", "-o", str(nii_path), str(dcm_path)],
        capture_output=True,
        text=True,
    )
    elapsed = round(time.time() - t0, 1)

    nii_files = list(nii_path.glob("*.nii.gz"))
    if result.returncode != 0 or not nii_files:
        msg = f"{uid}: return_code={result.returncode}\n{result.stderr}"
        dcm2niix_errors.append(msg)
        print(f"ERROR converting {uid}")
    else:
        usage_metrics["series"].setdefault(uid, {})["dcm2niix_s"] = elapsed
        converted_uids.append(uid)
        print(f"Converted {uid} -> {[f.name for f in nii_files]} ({elapsed}s)")

if dcm2niix_errors:
    Path("error_file.txt").write_text("\n".join(dcm2niix_errors))

print(f"\nSuccessfully converted {len(converted_uids)}/{len(series_uids)} series")

## Run MOOSE inference

In [ ]:
from moosez import moose

MOOSE_OUT_DIR = Path("/tmp/moose_output")
MOOSE_OUT_DIR.mkdir(parents=True, exist_ok=True)

moose_errors = []

for uid in converted_uids:
    nii_dir = NIFTI_DIR / uid
    out_path = MOOSE_OUT_DIR / uid
    out_path.mkdir(parents=True, exist_ok=True)

    # moosez Python API expects a NIfTI file path, not a directory
    nii_files = list(nii_dir.glob("*.nii.gz"))
    if not nii_files:
        msg = f"{uid}: no NIfTI files found in {nii_dir}"
        moose_errors.append(msg)
        print(f"ERROR: {msg}")
        continue
    nii_file = nii_files[0]

    t0 = time.time()
    try:
        moose(
            str(nii_file),
            models,
            str(out_path),
            accelerator,
        )
        elapsed = round(time.time() - t0, 1)
        usage_metrics["series"].setdefault(uid, {})["moose_s"] = elapsed

        seg_files = list(out_path.rglob("*.nii.gz"))
        print(f"MOOSE done for {uid}: {len(seg_files)} mask(s) in {elapsed}s")

    except Exception as exc:
        msg = f"{uid}: {traceback.format_exc()}"
        moose_errors.append(msg)
        print(f"ERROR running MOOSE on {uid}: {exc}")

if moose_errors:
    Path("moose_errors.txt").write_text("\n".join(moose_errors))

print(f"\nMOOSE completed for {len(converted_uids) - len(moose_errors)}/{len(converted_uids)} series")

## Package outputs

In [ ]:
def compress_dir(src_dir: Path, out_file: str) -> None:
    """Tar a directory and compress with lz4."""
    cmd = f"tar -cf - -C {src_dir.parent} {src_dir.name} | lz4 > {out_file}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Compression failed: {result.stderr}")
    size_mb = Path(out_file).stat().st_size / (1024 ** 2)
    print(f"Created {out_file} ({size_mb:.1f} MB)")


compress_dir(MOOSE_OUT_DIR, "moose_segmentations.tar.lz4")

## Write usage metrics

In [ ]:
import json

metrics_json = json.dumps(usage_metrics, indent=2)
metrics_path = Path("moose_UsageMetrics.json")
metrics_path.write_text(metrics_json)

# Compress metrics for Terra output (-f to overwrite if exists)
subprocess.run(
    f"lz4 -f {metrics_path} moose_UsageMetrics.lz4",
    shell=True,
    check=True,
)

print(metrics_json)

## Summary

In [ ]:
print("=" * 60)
print("MOOSE End-to-End Summary")
print("=" * 60)
print(f"  Total series        : {len(series_uids)}")
print(f"  Download failures   : {len(download_errors)}")
print(f"  dcm2niix failures   : {len(dcm2niix_errors)}")
print(f"  MOOSE failures      : {len(moose_errors)}")
print(f"  Models used         : {models}")
print("=" * 60)

# Fail the notebook (and therefore the papermill call) if every series failed
if len(moose_errors) == len(series_uids):
    raise RuntimeError("All series failed MOOSE inference — see moose_errors.txt")